# FuncCraft quick start

This notebook shows two things:

1. How to build one benchmark function from a plain Python `dict`.
2. How to build and use a `BenchmarkSuite` from a YAML suite spec.

The spec objects are intentionally shown as dictionaries so they can be
edited, printed, and later dumped to YAML/JSON without extra wrappers.


In [ ]:
import random
import sys
from pathlib import Path

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import numpy as np
import textwrap

sys.path.append(str(Path('..').resolve()))

from funccraft import BasicF, BasicFunctionId, BenchmarkFunction, BenchmarkSuite, load_suite_spec_yaml

def zeros(d):
    return [0.0] * d


## One function

The function spec is a nested dictionary. The keys mirror the benchmark
ingredients, but you do not need to construct helper classes yourself.


In [ ]:
func_spec = {
    'dimension': 10,
    'lower_bound': [-100.0] * 10,
    'upper_bound': [100.0] * 10,
    'component_specs': [
        {
            'base_function': 'Sphere',
            'component_dimension': 10,
            'coordinate_transform': {
                'kind': 'identity',
                'dimension': 10,
                'source_point': zeros(10),
                'target_point': zeros(10),
            },
            'value_transform': {
                'kind': 'identity',
            },
        }
    ],
    'composition_spec': {
        'kind': 'single_component',
    },
    'seed': 0,
    'function_class_label': 'C=NONE;P=NONE;T=NONE;G=Sphere',
    'known_global_minimizer': zeros(10),
    'known_global_value': 100.0,
}

f = BenchmarkFunction(func_spec)
print(f)
print('dimension:', f.dimension)
print('label:', f.spec.function_class_label)
print('spec as dict:', f.spec.to_dict())
print('f(x*):', f([f.spec.known_global_minimizer])[0])


## Benchmark suite

Here we generate 100 functions at dimension 2, then plot all of them
in batches of 24 per PDF page with wrapped titles.


In [ ]:
suite_spec = load_suite_spec_yaml(Path('..') / 'BenchmarkSuites' / 'default_suite.yaml')
suite_spec['supported_dimensions'] = '2'
suite_spec['requested_number_of_functions'] = 200
suite_spec['master_seed'] = 1
suite_spec['lower_bound'] = -100
suite_spec['upper_bound'] = 100
suite_spec['f_opt'] = 100.0

def wrap_title(idx, label):
    fields = {}
    for part in label.split(';'):
        if '=' in part:
            key, value = part.split('=', 1)
            fields[key.strip()] = value.strip()
    lines = [f'{idx}:']
    cpt = '; '.join(f'{key}={fields[key]}' for key in ('C', 'P', 'T') if key in fields)
    if cpt:
        lines.append(cpt)
    g_value = fields.get('G', '')
    if g_value:
        terms = g_value.split('+')
        if len(terms) <= 1:
            lines.append('G=' + g_value)
        else:
            best_split = 1
            best_score = None
            for split in range(1, len(terms)):
                left = '+'.join(terms[:split])
                right = '+'.join(terms[split:])
                score = max(len(left), len(right))
                if best_score is None or score < best_score:
                    best_score = score
                    best_split = split
            lines.append('G=' + '+'.join(terms[:best_split]))
            lines.append('+'.join(terms[best_split:]))
    return '\n'.join(lines)


In [ ]:
suite = BenchmarkSuite(suite_spec, 2)
print('size:', suite.size)
print('dimension:', suite.dimension)
print('max_number_of_functions:', suite.max_number_of_functions)
print('theoretical_max_number_of_functions:', suite.theoretical_max_number_of_functions)

selected_indices = list(range(len(suite)))
grid_x = np.linspace(-100.0, 100.0, 50)   
grid_y = np.linspace(-100.0, 100.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]


num_columns = 4
num_rows = 6
per_page = num_columns * num_rows
pdf_path = Path('benchmark_suite_100_dim2_log.pdf')

with PdfPages(pdf_path) as pdf:
    for page_start in range(0, len(selected_indices), per_page):
        page_indices = selected_indices[page_start:page_start + per_page]
        fig = plt.figure(figsize=(16, 24))
        for pos, idx in enumerate(page_indices, start=1):
            function = suite.function(idx)
            values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)

            values = np.log10(values )
            ax.view_init(elev=10, azim=45)

            zmin = float(np.nanpercentile(values, 5))
            zmax = float(np.nanpercentile(values, 60))
            ax = fig.add_subplot(num_rows, num_columns, pos, projection='3d')
            ax.plot_surface(xx, yy, values, cmap='viridis', linewidth=0.3, antialiased=True)
            ax.contour(xx, yy, values, zdir='z', offset=zmin, cmap='viridis', linewidths=0.5)
            ax.set_title(wrap_title(idx, function.spec.function_class_label), fontsize=6, pad=14)
            ax.set_xlabel('x1', fontsize=7)
            ax.set_ylabel('x2', fontsize=7)
            #ax.set_zlabel('f', fontsize=7)
            ax.set_zlabel('log10(f)', fontsize=7)
            #ax.set_zlim(zmin, zmax)
            #ax.set_zscale('log')

            ax.tick_params(labelsize=6)
            ax.view_init(elev=18, azim=-135)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}')
plt.show()


In [ ]:
dpm_suite_spec = suite_spec.copy()
dpm_suite_spec['composition_functions'] = [
    {'kind': 'cpmlwell', 'probability': 0.0, 'parameters': []},
    {'kind': 'cpmsum', 'probability': 0.0, 'parameters': []},
    {'kind': 'dpmsoftmax', 'probability': 1.0, 'parameters': [0.005]},
]

dpm_suite = BenchmarkSuite(dpm_suite_spec, 2)
dpm_indices = [49, 50, 57]
dpm_labels = ['F42', 'F45', 'F48']
grid_x = np.linspace(-100.0, 100.0, 50)
grid_y = np.linspace(-100.0, 100.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

fig = plt.figure(figsize=(18, 6))
for pos, (idx, label) in enumerate(zip(dpm_indices, dpm_labels), start=1):
    function = dpm_suite.function(idx)
    values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
    eps = 0.0
    values = np.log10(values )
    zmin = float(np.nanpercentile(values, 5))
    zmax = float(np.nanpercentile(values, 95))
    ax = fig.add_subplot(1, 3, pos, projection='3d')
    ax.plot_surface(xx, yy, values, cmap='viridis', linewidth=0.3, antialiased=True)
    ax.contour(xx, yy, values, zdir='z', offset=zmin, cmap='viridis', linewidths=0.5)
    ax.set_title(wrap_title(idx, f"{label}: {function.spec.function_class_label}"), fontsize=7)
    ax.set_xlabel('x1', fontsize=7)
    ax.set_ylabel('x2', fontsize=7)
    ax.set_zlabel('f', fontsize=7)
    #ax.set_zlim(zmin, zmax)
    #ax.set_zscale("log")
    ax.tick_params(labelsize=6)
    ax.view_init(elev=10, azim=45)

plt.tight_layout()
plt.show()


In [ ]:
dpm_suite_spec = suite_spec.copy()
dpm_suite_spec['composition_functions'] = [
    {'kind': 'cpmlwell', 'probability': 0.0, 'parameters': []},
    {'kind': 'cpmsum', 'probability': 0.0, 'parameters': []},
    {'kind': 'dpmbgsoftmax', 'probability': 1.0, 'parameters': [0.01, 1, 0.01]},
]

dpm_suite = BenchmarkSuite(dpm_suite_spec, 2)
dpm_indices = [49, 50, 57]
dpm_labels = ['F42', 'F45', 'F48']
grid_x = np.linspace(-100.0, 100.0, 50)
grid_y = np.linspace(-100.0, 100.0, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

fig = plt.figure(figsize=(18, 6))
for pos, (idx, label) in enumerate(zip(dpm_indices, dpm_labels), start=1):
    function = dpm_suite.function(idx)
    values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
    zmin = float(np.nanpercentile(values, 5))
    zmax = float(np.nanpercentile(values, 95))
    ax = fig.add_subplot(1, 3, pos, projection='3d')
    ax.plot_surface(xx, yy, values, cmap='viridis', linewidth=0.3, antialiased=True)
    ax.contour(xx, yy, values, zdir='z', offset=zmin, cmap='viridis', linewidths=0.5)
    ax.set_title(wrap_title(idx, f"{label}: {function.spec.function_class_label}"), fontsize=7)
    ax.set_xlabel('x1', fontsize=7)
    ax.set_ylabel('x2', fontsize=7)
    ax.set_zlabel('f', fontsize=7)
    #ax.set_zlim(zmin, zmax)
    #ax.set_zscale("log")
    ax.tick_params(labelsize=6)
    ax.view_init(elev=18, azim=-135)

plt.tight_layout()
plt.show()


## Pure base functions

This page plots the raw base functions directly, without any coordinate
transform or value transform. The layout matches the benchmark-suite
page style, and paginates 24 functions per PDF page.
Each plot uses the primitive's `default_domain`.


In [ ]:
base_function_names = [
    'Sphere',
    'SumDifferentPowers',
    'Ellipsoidal',
    'BuecheRastrigin',
    'LinearSlope',
    'AttractiveSector',
    'StepEllipsoidal',
    'StepRastrigin',
    'Rosenbrock',
    'Ackley',
    'Rastrigin',
    'Griewank',
    'Schwefel',
    'SharpRidge',
    'Weierstrass',
    'SchafferF7',
    'SchafferF7Cond1000',
    'GriewankRosenbrock',
    'Gallagher21',
    'Katsuura',
    'LunacekBiRastrigin',
    'Zakharov',
    'Levy',
    'Michalewicz',
    'DixonPrice',
    'BentCigar',
    'Discus',
    'HappyCat',
    'HGBat',
    'HCF',
    'GrieRosen',
    'SchafferF6',
    'Step',
    'Quartic',
    'Exponential',
    'StyblinskiTang',
]

default_domain = BasicF(BasicFunctionId.Sphere, 2).default_domain
grid_x = np.linspace(-2, 1, 50)
grid_y = np.linspace(-2, 1, 50)
xx, yy = np.meshgrid(grid_x, grid_y)
grid_points = [[float(x), float(y)] for x, y in zip(xx.ravel(), yy.ravel())]

def wrap_title(idx, label, width=14):
    wrapped = textwrap.wrap(label, width=width, break_long_words=False, break_on_hyphens=False)
    return f"{idx}:\n" + "\n".join(wrapped)

pdf_path = Path('pure_base_functions_dim2.pdf')
num_columns = 4
num_rows = 6
per_page = num_columns * num_rows

with PdfPages(pdf_path) as pdf:
    for page_start in range(0, len(base_function_names), per_page):
        page_names = base_function_names[page_start:page_start + per_page]
        fig = plt.figure(figsize=(16, 24))
        for local_pos, base_name in enumerate(page_names, start=1):
            global_pos = page_start + local_pos - 1
            function = BasicF(getattr(BasicFunctionId, base_name), 2)
            values = np.asarray(function(grid_points), dtype=float).reshape(xx.shape)
            zmin = float(np.nanpercentile(values, 5))
            zmax = float(np.nanpercentile(values, 95))
            ax = fig.add_subplot(num_rows, num_columns, local_pos, projection="3d")
            try :
                ax.plot_surface(xx, yy, values, cmap="viridis", linewidth=0.3, antialiased=True)
            except : 
                continue
            ax.contour(xx, yy, values, zdir="z", offset=zmin, cmap="viridis", linewidths=0.5)
            ax.set_title(wrap_title(global_pos, base_name), fontsize=5, pad=12)
            ax.set_xlabel("x1", fontsize=7)
            ax.set_ylabel("x2", fontsize=7)
            ax.set_zlabel("f", fontsize=7)
            ax.set_zlim(zmin, zmax)
            ax.tick_params(labelsize=6)
            ax.view_init(elev=18, azim=-135)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Saved {pdf_path}")
